# AWS Certified Data Engineer Associate — Cheat Sheet

> **Quick-reference notes** organized by exam domain. Each bullet is exam-ready.

| Domain | Weight |
|--------|--------|
| Domain 1: Data Ingestion and Transformation | ~34% |
| Domain 2: Data Store Management | ~26% |
| Domain 3: Data Operations and Support | ~22% |
| Domain 4: Data Security and Governance | ~18% |

---


# Domain 1: Data Ingestion and Transformation

### EventBridge vs SNS vs SQS

- **EventBridge** — SaaS integration, 90+ AWS services as event sources, JSON-based routing rules, ~15 target types; use when source is a third-party SaaS app
- **SNS** — fan-out to many subscribers (Lambda, SQS, HTTP); push-based; no SaaS integration
- **SQS Standard** — decouple one consumer at a time; valid S3 event notification target
- **SQS FIFO** — exactly-once, ordered; **NOT** a valid S3 event notification target
- **ELB** — synchronous decoupling only; not for async/event-driven patterns

Key rules:
- SaaS third-party integration → always **EventBridge** (SNS cannot do this)
- S3 → single EC2 processes each upload → use S3 Event Notification → **SQS Standard** → EC2 polls queue
- S3 → SNS notifies ALL subscribers simultaneously (wrong for single-consumer scenarios)

### Amazon Kinesis Data Streams (KDS)

- Massively scalable real-time streaming; data available in **milliseconds**
- Ordered records per partition key (shard); each shard: **1 MB/s write, 2 MB/s read**
- **Resharding** — split shards to increase capacity
- **ProvisionedThroughputExceededException** → use `PutRecords` API to batch messages; increasing shards is expensive
- **Duplicate records** — caused by producer network timeout + retry; both writes succeed → embed a primary key to deduplicate downstream
- **GetRecords.IteratorAgeMilliseconds** — high value means consumer is falling behind; fix by adding consumers or shards

**Kinesis Agent**
- Stand-alone Java app; writes to KDS or Firehose
- Cannot write to Firehose when Firehose source is already set to a KDS stream (PutRecord/PutRecordBatch disabled in that mode)

**Kinesis Data Firehose**
- Fully managed, auto-scales, single destination per stream (S3, Redshift, OpenSearch, Splunk)
- Cannot fan-out to multiple consumers — use KDS for that
- Lambda can transform records inside Firehose before delivery

### AWS Glue

- Serverless ETL; runs Apache Spark and Python shell jobs
- **Glue Crawler** — auto-detects schema; writes to Glue Data Catalog
  - Partition threshold: **> 70%** same schema → 1 merged table; ≤ 70% → separate tables per folder
  - Max distinct schemas (clusters) per folder: **5**
  - Example: 8/10 files same schema = 80% → 1 table; 7/10 = 70% → 2 tables
- **Glue DataBrew** — visual no-code data prep; 250+ transformations; supports CSV, Excel, JSON, ORC, Parquet
  - Athena does NOT support Excel — use DataBrew for Excel files
- **Glue ETL job** — converts formats (e.g., CSV → Parquet), partitions output for Athena performance

### AWS DMS + Schema Conversion Tool (SCT)

- **Homogeneous migration** (Oracle → Oracle): DMS alone
- **Heterogeneous migration** (Oracle → Aurora, SQL Server → MySQL): **SCT first** then DMS
  - SCT handles: schema, stored procedures, secondary indexes, foreign keys
  - Basic Schema Copy in DMS does NOT handle secondary indexes, foreign keys, or stored procedures
- DMS supports continuous replication with HA; can stream to Redshift and S3

### KDS + AWS Fargate (ECS) for Long-Running Jobs

- Lambda max timeout = **15 minutes**; use **ECS Fargate** for jobs that run longer
- Fargate = serverless containers; no EC2 management; scales automatically

### Snowball Edge

- For petabyte-scale offline transfer: **Snowball Edge Storage Optimized** (up to 80 TB each)
- Cannot copy directly from Snowball Edge to S3 Glacier — must go: Snowball → S3 → Lifecycle policy → Glacier

### IoT / Notifications Pattern

- Real-time IoT streaming → **KDS** (ingest) + **SNS** (push notifications to mobile apps)
- SES = email only; SQS = polling queue (no push); SNS = push pub/sub to mobile/Lambda/HTTP

### Lambda Limits

| Limit | Value |
|-------|-------|
| Max execution timeout | **15 minutes** |
| Ephemeral /tmp storage | 512 MB – 10,240 MB |
| Max layers per function | **5** (zip deployments only; container images do not support layers) |
| Lambda + Kinesis | Ordered per partition key; up to 10 concurrent batches per shard |

## Domain 1 — Additional Topics (Set 3)

### KDS → KDA → Firehose Pipeline
- Real-time analytics pattern: **KDS → Kinesis Data Analytics (KDA) → Firehose → S3**
- QuickSight **cannot** use KDS as a source; Athena **cannot** analyze data in real-time
- Firehose = near real-time (not real-time); use KDA for real-time SQL/Flink analytics

### Amazon Managed Service for Apache Flink (formerly KDA)
- Supports **time-based window aggregations up to 45 minutes** with high fault tolerance
- Fully managed; scales automatically; no servers to manage
- Use when you need real-time analytics with complex windowing (not Lambda, not Firehose)
- KDA only accepts **KDS or Firehose** as streaming sources (cannot write Lambda output directly to KDA)

### Kinesis Data Streams — Concurrent Consumers
- KDS = **multiple applications** can consume the same stream **concurrently and independently**
- Firehose = single destination delivery only
- SQS = message consumed by **one** consumer at a time (no concurrent consumption of same message)
- Use KDS when: ordering required, multiple consumers, replay capability needed

### KDS Partition Key Design
- Use **high-cardinality keys** (e.g., sensor ID) to evenly distribute data across shards
- Low-cardinality keys (e.g., vehicle name) concentrate data on one shard → hot shard bottleneck
- Fix hot shards by changing partition key; no need to add shards if total throughput is within limits

### Amazon Kinesis Data Firehose
- Fully managed, **auto-scales**, no ongoing administration
- Destinations: S3, Redshift, Elasticsearch, Splunk
- Best for: log file ingestion into persistent storage with least overhead
- Small files issue: if delivery stream **scales up**, buffer size halves → more parallel files created in S3
- Cannot deliver to multiple S3 buckets from a single stream

### Amazon AppFlow — SaaS Ingestion
- Fully managed service to transfer data from **SaaS apps** (Zendesk, Salesforce, Marketo, Datadog) → S3 or Redshift
- Low-code/no-code; **least operational overhead** for SaaS → AWS data lake patterns
- Not Kinesis (requires custom code); not DataBrew (cannot read from SaaS apps)

### S3 Event Notifications for CSV Processing
- Use event type `s3:ObjectCreated:*` (not `s3:*` which fires for ALL S3 events)
- Add **suffix filter** for `.csv` to trigger only on CSV uploads
- Set destination to **Lambda ARN directly** — no need for EventBridge indirection unless you need all events delivered to EventBridge

### S3 Multipart Upload + Transfer Acceleration
- For large files (>100 MB) over long distances: use **multipart upload + S3 Transfer Acceleration**
- Multipart = parallel parts → maximizes bandwidth; Transfer Acceleration = CloudFront edge routing
- Multipart alone is good; combining with TA gives maximum speed

### Glue Python Shell vs PySpark
- **Python Shell job**: 0.0625 DPU (1/16 DPU), Pandas/NumPy/SciPy pre-loaded, no Spark overhead
- Most **cost-effective** for small-to-medium ETL (CSV files ~100 MB)
- PySpark = more powerful but costlier; use for large-scale distributed workloads

### Glue DataBrew Transformations
- **NEST_TO_MAP**: selected columns → JSON key-value pairs (column name = key, row value = value); **order NOT maintained**
- **NEST_TO_ARRAY**: selected columns → array values; **order IS maintained**
- Athena **cannot** query Microsoft Excel (`.xls`) files — supported formats: CSV, JSON, ORC, Parquet, Avro, CloudTrail logs, WebServer logs

### Athena Federated Queries + PartiQL
- Athena supports federated queries across RDS, DynamoDB, DocumentDB, and JDBC-compatible sources
- Use **PartiQL** syntax to query DynamoDB via Athena federated query pass-through
- Prebuilt connectors exist for most AWS data sources; runs on Lambda

### SQL Regex in Redshift/PostgreSQL
- `~` = case-sensitive regex match
- `~*` = **case-insensitive** regex match
- `^` = match at **start** of string; `$` = match at **end** of string
- Example: `WHERE name ~* '^(John|Doe)'` → case-insensitive match for names starting with John or Doe


## Domain 1 – Set 5: Data Ingestion and Transformation

**S3 Transfer Acceleration + CloudFront for remote offices:**
- S3TA: speeds uploads AND downloads 50–500% for long-distance transfers; routes via CloudFront edge locations
- CloudFront CDN: also speeds up downloads by caching at edge locations
- Both are valid; S3TA ≠ CloudFront: S3TA = direct accelerated path to S3; CloudFront = caches content

**MSK vs KDS vs Firehose for large records:**
- KDS max record = 1 MB; Firehose max = 1,000 KB (~1 MB); MSK (Kafka) max = **100 MB**
- Use MSK when records range 500 KB – 10 MB (exceeds KDS/Firehose limits)

**KDS ProvisionedThroughputExceeded → batch messages:**
- Fix: use `PutRecords` API to batch multiple records per request (reduces overhead, increases throughput)
- Increasing shards = costly; Exponential Backoff = short-term only; Decreasing retention = data loss

**API Gateway + Kinesis Data Analytics for real-time streaming:**
- REST API (API GW) receives location/IoT data → KDA processes stream in real-time
- Athena = queries static S3 data (not streaming); QuickSight = BI dashboards (no REST API ingest)

**SQS FIFO batch mode throughput:**
- Default: 300 msg/sec | Batch 10: 3,000/sec | **Batch 4: 1,200/sec** | Batch 2: 600/sec
- Use batch mode when FIFO ordering is required but 300/sec default is insufficient

**CloudWatch Logs cross-account aggregation (near-real-time):**
- Set up Kinesis Data Firehose in logging account → subscribe to CWL streams in each app account via **subscription filters** → store in S3
- CWL agent cannot push directly to Firehose; Lambda hourly export = not near-real-time

**SQS + Lambda for fully serverless IoT pipeline:**
- SQS Standard Queue + Lambda event source mapping = fully serverless, auto-scaling, no capacity planning
- Firehose cannot write directly to DynamoDB; EC2-based solutions = not serverless
- Lambda processes messages in batches from SQS automatically

**Redshift streaming ingestion from KDS (lowest latency):**
- Direct path: KDS → Redshift **materialized view** (bypasses Firehose, bypasses S3 COPY)
- Refresh materialized view to ingest hundreds of MB/sec with near-real-time latency
- Redshift federated queries and Spectrum cannot process streaming data

**DynamoDB for IoT key-value data + change streaming:**
- DynamoDB: key-value + document NoSQL; handles variable/evolving schemas; single-digit ms latency
- DynamoDB Streams: ordered flow of item-level changes (create/update/delete) for downstream processing

**Snowmobile vs Snowball:**
- Snowmobile: up to 100 PB per truck; AWS recommends for **≥ 10 PB** in a single location
- Snowball: for < 10 PB; DataSync = online (poor for remote/slow internet); Direct Connect = slow to provision

**Athena SQL on S3 (sanity checks, ad-hoc queries):**
- Serverless, no infrastructure to manage, pay-per-query → most cost-effective for ad-hoc S3 analysis
- vs RDS/Redshift/EMR: all require cluster/instance provisioning and ongoing management

**KDS Enhanced Fan-Out for multiple parallel consumers:**
- Default: 2 MB/sec/shard **shared** across all consumers
- Enhanced Fan-Out: each registered consumer gets a **dedicated** 2 MB/sec/shard (push-based)
- Use when multiple applications consume the same stream concurrently and need independent throughput

## Domain 1 – Data Ingestion and Transformation (Set 7)

### Redshift Streaming Ingestion (KDS → Redshift direct)
- **Setup**: CREATE EXTERNAL SCHEMA mapping to Kinesis stream → CREATE MATERIALIZED VIEW referencing external schema → set `AUTO REFRESH YES`
- **Key advantage**: bypasses S3 staging and COPY command entirely; latency in **seconds**, not minutes
- **Why not Firehose→S3→COPY?** Adds latency + operational overhead — ruled out when "low latency" is required
- You **cannot** create a materialized view directly on a KDS stream without the external schema

```
Redshift Streaming Ingestion:
┌──────────────┐    ┌──────────────────────┐    ┌──────────────────────┐
│ Kinesis Data │───▶│  EXTERNAL SCHEMA     │───▶│  MATERIALIZED VIEW   │
│   Streams    │    │  (maps to KDS source)│    │  (auto-refresh)      │
└──────────────┘    └──────────────────────┘    └──────────┬───────────┘
  No S3 staging                                             │
  No COPY cmd                                               ▼
  Seconds latency                                  [Redshift / BI Tools]
```

### SQS FIFO Throughput Formula
- Default: **300 API calls/sec** (send/receive/delete)
- With batching (max 10 msgs/call): **300 × batch_size = throughput**
- Must use FIFO (not standard) when **order** is required

```
SQS FIFO Batch Throughput:
  Batch  4 × 300 =  1,200 msg/sec
  Batch  8 × 300 =  2,400 msg/sec  ← answer when peak = 2,400
  Batch 10 × 300 =  3,000 msg/sec  (MAXIMUM — batch > 10 is invalid)
```
- Batch of 12 is **invalid** (max is 10); batch of 4 only gives 1,200 (insufficient for 2,400)

### SQS FIFO Group ID — Scaling Multiple Consumers
- Without `MessageGroupId` → all messages in strict global order → **only 1 consumer**
- With `MessageGroupId = deviceID` → per-group ordering → **1 consumer per group** → scales to N devices
- Use case: telemetry from many desktop systems where each device's data must be ordered independently

### KDS — Ordered Processing + Replay
- Set **Partition Key = entity/device ID** → same key always routes to same shard → ordering guaranteed per key
- KDS stores records for up to **365 days** → multiple consumers (billing app + audit app) can read same data in same order hours/days later
- **KDS vs SQS**: SQS cannot replay messages to multiple consumers in order → use KDS when replay/audit needed

### Kinesis Data Firehose — Lambda Transform Pattern
- Firehose has **native plug-and-play Lambda integration** for transformation before delivery
- Flow: Source → Firehose → Lambda (filter/transform) → S3 / Redshift / OpenSearch
- Fully managed, auto-scales, no infrastructure — preferred over KDS+Lambda for "least infrastructure maintenance"
- **Firehose → Redshift**: delivers to S3 first, then issues COPY command automatically; near-real-time; fully managed

### DMS: S3 → Kinesis Data Streams
- **Fastest** way to stream existing + new S3 files to KDS without custom code
- DMS supports S3 as **source** and KDS/MSK as **target**
- Supports full load + CDC (change data capture)
- No complex config; scales via replication instance size

### Multipart Upload
- Recommended for objects **≥ 100 MB**; **required** for objects **> 5 GB**
- Upload parts independently and in parallel → higher throughput + fault-tolerant retries
- If a part fails, only that part is retransmitted

### EMR vs Glue for Spark Jobs
| Feature | Amazon EMR | AWS Glue |
|---|---|---|
| Spark support | ✅ | ✅ |
| Access to EC2 | ✅ (direct) | ❌ (serverless) |
| Infrastructure mgmt | Required | None |
| Choose when | More control needed | "Least overhead" / serverless |

### Glue Job Performance Optimization
- **Partition S3 data** by year/month/day → crawlers and jobs only scan relevant partitions
- **Use larger worker types** (G.1X, G.2X) → more cores/memory → vertical scale, fewer shuffles
- NOT: checking IAM permissions, streaming mode, or cross-job clusters (distractors)

### S3 Transfer Acceleration + Multipart (for global uploads)
- **S3TA**: routes uploads through CloudFront edge → optimized path to S3 bucket (overseas branches)
- **Multipart upload**: improves throughput + resilience for large files
- Use **both together** for large files uploaded from remote geographies

# Domain 2: Data Store Management

---

## Amazon Athena — Performance & Format

### File Formats
| Format | Use | Notes |
|--------|-----|-------|
| **Parquet** | Columnar — best for analytical queries (select few columns) | Preferred for Athena |
| **ORC** | Columnar — similar to Parquet; smaller files | Also good but no compression specified separately |
| **CSV / TSV / JSON** | Row-based — poor Athena performance | Default, no optimization |
| **ZIP** | ❌ Not supported by Athena | |
| **Excel (.xls)** | ❌ Not supported by Athena | Use Glue DataBrew instead |
| **Avro** | Row-based — better for ETL/full-row operations | Not ideal for column queries |

### Compression
- **Snappy** — fast compression/decompression (LZ77 family); best for Athena query speed
- Also supported: gzip, zstd

### Partitioning
- Restrict data scanned → lower cost + faster queries
- **Hive-compatible format**: `date=2024-01-15/` (key=value) → use `MSCK REPAIR TABLE` to load
- **Non-Hive format**: `2024/01/15/` → use `ALTER TABLE ADD PARTITION` manually
- The exam uses `MSCK REPAIR TABLE` when partition keys follow Hive style

### JOIN Optimization
> Put the **larger table on the LEFT**, smaller table on the RIGHT — Presto streams left, distributes right in memory.

### Cross-Region
- Possible but incurs extra data transfer charges → **always co-locate S3 bucket in the same region as Athena**
- S3 is **regional**, not Availability Zone-level — cannot configure at AZ level

### Athena Workgroups
- Separate teams, control data scanned per query, isolate query history
- Use **resource-level IAM policies** on workgroups — Lake Formation does NOT control Athena query history

### Athena Federated Query
- Query data across **DynamoDB, RDS, DocumentDB, CloudWatch Logs, MySQL, PostgreSQL** via Lambda connectors
- No need to copy data to S3 first — most cost-effective for multi-source joins

---

## Amazon S3 Storage Classes

### Lifecycle Transition Rules (Waterfall — can only go DOWN)
```
Standard → Intelligent-Tiering → Standard-IA → One Zone-IA → Glacier Flexible → Glacier Deep Archive
```

**INVALID transitions (exam trap):**
- ❌ Any class → **Standard** (cannot transition back up)
- ❌ Any class → **Reduced Redundancy**
- ❌ **Intelligent-Tiering** → Standard-IA
- ❌ **One Zone-IA** → Standard-IA or Intelligent-Tiering

**Minimum storage duration before transition:**
- Standard → Standard-IA or One Zone-IA: **30 days minimum**
- Cannot transition after only 7 days

### Storage Class Quick Reference
| Class | When to use | Key fact |
|-------|-------------|----------|
| Standard | Frequent access | Default; multi-AZ |
| Intelligent-Tiering | Unknown access patterns | Auto-moves between tiers |
| Standard-IA | Infrequent, needs fast retrieval, resilient | Multi-AZ; higher durability |
| **One Zone-IA** | Infrequent, **re-creatable** data | Single AZ; 20% cheaper than Std-IA; 30-day min |
| Glacier Flexible | Archive, 1-5 min expedited retrieval | |
| Glacier Deep Archive | Long-term archive, 12-hr retrieval | Lowest cost |

> Re-creatable assets accessed heavily first then rarely → **One Zone-IA** after 30 days (not 7).

### S3 Analytics (Storage Class Analysis)
- Analyzes access patterns to recommend transitions
- Only recommends **Standard → Standard-IA** (no other class pairs)
- Does NOT recommend Glacier, One Zone-IA, or Intelligent-Tiering

### S3 Event Notifications
- Destinations: **Lambda, SNS, SQS Standard** only
- **SQS FIFO is NOT supported** as S3 event notification destination
- Use SQS Standard when only ONE of N consumers should process each upload

---

## Amazon DynamoDB

### Partition Formula
$$\text{Partitions} = \max\left(\left\lceil\frac{\text{Storage (GB)}}{10}\right\rceil,\ \left\lceil\frac{\text{WCU}}{1000} + \frac{\text{RCU}}{3000}\right\rceil\right)$$

> Example: 500 WCU, 5000 RCU, 50 GB → max(⌈50/10⌉, ⌈0.5+1.67⌉) = max(5, 3) = **5 partitions**

### Secondary Indexes
| Feature | LSI (Local Secondary Index) | GSI (Global Secondary Index) |
|---------|----------------------------|------------------------------|
| Created | **At table creation only** — cannot add later | At creation OR added/deleted later |
| Key | Same partition key, different sort key | Completely different partition + sort key |
| Consistency | Strongly OR eventually consistent | **Eventually consistent only** |
| Max per table | **5** | 20 (default quota) |
| Throttling | Independent | If GSI writes throttled → **main table also throttled** |

### DynamoDB Streams + Lambda
- Ordered flow of item changes (create/update/delete)
- Lambda triggered on each change — zero-delay reaction (e.g., congratulate on milestone)
- EventBridge **cannot** use DynamoDB as a target

### DynamoDB TTL (Time to Live)
- Add a numeric attribute (epoch timestamp) → enable TTL → DynamoDB auto-deletes expired items
- **Free** — no write throughput consumed
- Minimum development effort vs. Lambda/Glue/EC2 cron approaches

### DynamoDB Conditional Writes
- `PutItem`, `UpdateItem`, `DeleteItem` succeed **only if** specified condition is met
- Prevents overwriting existing items with same primary key

### DynamoDB Accelerator (DAX)
- In-memory cache; upgrades DynamoDB from milliseconds → **microseconds**
- API-compatible — minimal code changes
- Best for: IoT, leaderboards, high-throughput read-heavy workloads
- Use **DynamoDB + DAX** when microsecond latency is required

### DynamoDB + SageMaker
- Use **boto3 DynamoDB client** `.scan()` from SageMaker notebook — direct, no S3 intermediary needed

---

## Amazon Redshift

### Concurrency Scaling
- Supported on **RA3 nodes only** (not DC2)
- Enable: set **WLM queue → Concurrency Scaling mode = auto**
- Scales read AND write automatically

### Short Query Acceleration (SQA)
- Prioritizes short queries ahead of long ones
- Does **NOT** affect write operations

### Materialized Views
- Precomputed result sets for complex/repeated queries
- Schedule refresh: **Redshift Query Editor v2** (internally uses EventBridge + Redshift Data API)
- `auto-refresh` refreshes whenever base table changes — not on a schedule

### Redshift Spectrum
- Query S3 data in-place from Redshift — does NOT copy data into Redshift

---

## Amazon EBS Volume Types

| Type | IOPS | Use case |
|------|------|----------|
| **io1/io2** (Provisioned IOPS SSD) | Up to 64,000 | I/O-intensive DBs; consistent performance |
| **gp2/gp3** (General Purpose SSD) | Up to 16,000 | Cost-effective; burst up to 3,000 IOPS |
| **st1** (Throughput HDD) | Up to 500 | Big data, Kafka, log processing |
| **sc1** (Cold HDD) | Up to 250 | Infrequently accessed, archival |

> io1 is expensive and for sustained high IOPS. gp2 is cheaper and bursts — better when workload is under-utilized with occasional spikes.
> EBS io1 = 90% of cost, under-utilized with occasional bursts → convert to **gp2**.

---

## Amazon EFS (Elastic File System)

- NFS-based file system for Linux — can **mount** on EC2 instances
- **EFS Infrequent Access (EFS IA)**: up to 92% cheaper than EFS Standard for files not accessed daily
- Multi-AZ by default (S3 Glacier and S3 IA are not mountable file systems)
- Use EFS IA when files are frequently accessed at first, then rarely — and you need a mountable NFS

---

## ElastiCache Redis vs DynamoDB + DAX

| | ElastiCache Redis | DynamoDB + DAX |
|--|------------------|----------------|
| Latency | Sub-millisecond | Microsecond |
| Use case | Leaderboards, sessions, real-time analytics, caching | DynamoDB-backed apps needing cache layer |
| In-memory | Yes | Yes |
| Data model | Flexible (sorted sets for leaderboards) | Key-value |

> Both are valid for gaming leaderboards. DynamoDB is **not** in-memory by itself — needs DAX.

---

## Static Content Distribution

- **Amazon S3** → host static websites (HTML, CSS, JS, images)
- **CloudFront + S3** → CDN; reduces latency, reduces ECS/EC2 load, cheaper egress
- Dynamic content (server-side scripting) cannot be served from S3
- EFS stores files but does NOT distribute them — still needs server to serve

---

## AWS Data Exchange

- Discover, subscribe to, and use **third-party datasets** directly within AWS
- Integrates natively with AWS analytics services (Athena, S3, Redshift)
- Lower overhead than DataSync or Marketplace for dataset integration


## Domain 2 — Additional Topics (Set 3)

### S3 Intelligent-Tiering
- **Default tier behavior**: Frequent Access → Infrequent Access after 30 days → Archive Instant Access after 90 days
- **Cannot manually** select Archive Instant Access tier — it is automatic only
- Best for data with unknown/unpredictable access patterns (data lakes, new applications)
- No retrieval fees when objects move between tiers; small per-object monitoring charge

### S3 Strong Consistency
- Amazon S3 provides **strong read-after-write consistency** automatically — no additional cost
- Overwrite an object then read immediately → **always returns latest version**
- Applies to GET, PUT, LIST operations, and tag/ACL/metadata changes

### S3 Standard for Short-Lived Staging Data
- S3 Standard = **no minimum storage duration**, no retrieval fee
- Best for frequently accessed or short-lived data (e.g., 24-hour staging zones)
- S3 Standard-IA and One Zone-IA have 30-day minimum → costly for sub-30-day staging
- S3 Glacier Instant Retrieval has 90-day minimum → wrong for short-lived data

### DynamoDB Global Tables
- **Multi-region, multi-master** replication; low-latency local reads and writes globally
- Supports **variable schema** (NoSQL); max item size **400 KB**
- Best for: ad-tech, IoT, global applications needing millions of requests/sec
- DocumentDB = single-region primary (read replicas only in secondary); RDS = not high-throughput

### DynamoDB + DAX for IoT
- DynamoDB = fully managed NoSQL with single-digit millisecond performance
- **DAX** (DynamoDB Accelerator) = in-memory cache built into DynamoDB; microsecond latency
- Use DAX when access patterns are known and read volume is very high
- ElastiCache is a general-purpose cache layer; DAX is DynamoDB-specific

### Caching Layer Options
- **DAX** = DynamoDB-specific in-memory cache
- **ElastiCache** (Redis/Memcached) = general caching layer for RDS, DynamoDB, or custom apps
- RDS, Redshift, Elasticsearch = **NOT** caching layers

### Glue Partition API — Lowest Latency Sync
- **Inline `create_partition` Boto3 API call** from data-writing code = lowest latency to update Glue Data Catalog
- Alternatives with higher latency: Glue Crawler on schedule, MSCK REPAIR TABLE (manual), Lambda on schedule

### Athena CTAS: Partitioning vs Bucketing
- **Partitioning** = low-cardinality columns (e.g., department, region, date) — limits number of partitions
- **Bucketing** = high-cardinality columns with **evenly distributed** values (e.g., timestamps, user IDs)
- Buckets by columns with sparse/skewed values = bad choice; use even distribution
- Can combine partitioning and bucketing in the same CTAS query

### EFS Provisioned Throughput
- Use when **throughput requirements are high relative to the amount of stored data**
- Example: development tools, web serving, content management with small data but high I/O needs
- Bursting Throughput scales with storage size — not ideal when storage is small
- EFS has higher throughput than EBS and is shared across multiple EC2/ECS/Lambda instances

### EFS Infrequent Access (EFS IA)
- **POSIX-compliant NFS** file system — use when POSIX compliance is required
- Up to **92% lower cost** than EFS Standard for files not accessed daily
- Suitable for archiving on-premises file data that is rarely accessed

### Redshift Serverless
- Auto-provisions and scales capacity automatically; **pay only when in use**
- No cluster management needed; ideal for **monthly or infrequent** workloads
- Better than provisioned cluster + EventBridge/Step Functions automation for low-frequency use

### Redshift Column-Level Access Control
- Use `GRANT SELECT (col1, col2) ON TABLE t TO GROUP finance` for column-level permissions
- No need to create views or duplicate tables
- Available since March 2020 in all Redshift regions

### Redshift Audit Logging
- Only **SSE-S3 (AES-256)** encryption supported for Redshift audit logs — **cannot use SSE-KMS**
- Use **Redshift Spectrum** to query audit logs in S3 for monthly audits (no need to COPY data back)

### Redshift Spectrum for Historical Data
- Query S3 data **directly from Redshift** without loading — Redshift Spectrum creates external tables
- Pushes predicate filtering/aggregation to Spectrum layer; less cluster resource usage
- Best for cross-referencing cold S3 data with hot Redshift data

### DMS for RDS Archival to S3
- Use **AWS DMS task** (periodic schedule) to migrate data older than 3 months from RDS → S3
- Add **S3 Glacier Deep Archive lifecycle rule** for long-term cost optimization
- Do NOT use Lambda with RDS Events (events don't fire for data changes); avoid Step Functions+DataSync (complex)

### RDS Performance Insights
- Identifies **slow SQL queries** and DB load broken down by: waits, SQL statements, hosts, users
- Central metric: **DB Load** (average active sessions per second)
- Use for: identifying which queries cause CPU spikes in RDS MySQL/PostgreSQL/etc.

### RDS Enhanced Monitoring
- Captures **OS-level** metrics in real-time: CPU usage per process/thread, memory, filesystem, disk I/O
- Stored in CloudWatch Logs (default 30 days)
- Different from standard CloudWatch metrics which are hypervisor-level

### Athena Cold Data Strategy
- Move infrequent data to **S3 Standard-IA** after 30 days + use **Athena** to query
- Athena = serverless, no infrastructure, pay-per-query — preserves SQL querying capability
- Do NOT create a smaller Redshift cluster (costs grow with storage; no tiered storage in Redshift)

### Snowball Edge Clustering
- **Only Snowball Edge** (both Compute Optimized and Storage Optimized) support **storage clustering**
- Snowcone = no clustering; Snowmobile = no clustering
- Clustered Snowball Edge devices can be rack-mounted for temporary large-scale installations


## Domain 2 – Set 5: Data Store Management

**Athena join order optimization:**
- Larger table on **LEFT**, smaller table on **RIGHT** → Athena builds lookup hash table from RIGHT
- Smaller right table = less memory = faster distributed hash join
- Engine version settings are distractors for this issue

**Athena table location must use per-table prefixes:**
- Zero results if: `s3://bucket/file.csv` (no prefix/folder)
- Correct: `s3://bucket/table1/table1.csv` (each table needs its own folder/prefix)
- Also causes zero results: double-slash paths (`//`), filenames starting with `_` or `.`

**S3 vs EFS cost:**
- S3 = most cost-effective object storage for large datasets (images, media)
- EFS = costlier (NFS file system); SageMaker instances ~40% costlier than EC2
- Use EC2 + S3 for cost-optimal ML inference pipelines

**S3 Intelligent-Tiering:**
- Auto-moves objects between frequent/infrequent access tiers based on 30-day access pattern
- No retrieval fee; small monthly monitoring fee per object
- Use when access patterns are **unknown or changing**; do NOT use when patterns are predictable/well-known

**Amazon Neptune for graph queries:**
- Purpose-built graph DB for highly connected datasets; optimized for billions of relationships
- Use cases: social networks (friends-of-friends likes), fraud detection, recommendations, knowledge graphs
- Not OpenSearch (search/logs), not Redshift (data warehouse), not Aurora (RDBMS)

**S3 Standard-IA for infrequent but rapid-access data:**
- Millisecond access when needed; lower storage cost + per-GB retrieval fee
- 99.9% availability (vs 99.99% S3 Standard); use retry/failover logic to compensate
- Best for: audit reports (2×/year), compliance archives queried occasionally

**EC2 Instance Store for highest IOPS temporary storage:**
- Physically attached NVMe/SSD; highest random I/O and lowest latency of all EBS options
- Included in instance cost (no extra charge) → most cost-optimal for temp processing
- Ephemeral: data lost on stop/terminate; use for buffers, caches, scratch data before writing to S3

**RDS for ACID workloads:**
- Supports Atomic, Consistent, Isolated, Durable transactions; strong read-after-write consistency
- Best when: frequent overwrites/deletes and latest data must always be returned
- Neptune = graph; ElastiCache = caching layer; S3 = object storage (not transactional DB)

**RDS → S3 → Redshift + Redshift Spectrum pattern:**
- Data Pipeline: incremental RDS → S3 copy (automated)
- Load last 6 months into Redshift for high-performance frequent queries
- Redshift Spectrum: query full historical data in S3 without loading (cost-effective cold data access)

**EBS volume types:**
- **io1** (Provisioned IOPS SSD): high-IOPS databases (PostgreSQL, MySQL, Oracle, MongoDB) — supports >16,000 IOPS
- **gp2** (General Purpose SSD): up to 16,000 IOPS; general workloads
- **st1** (Throughput Optimized HDD): big data, data warehouses (high throughput, not IOPS)
- **sc1** (Cold HDD): lowest cost; infrequently accessed data
- io1 and gp2 = persistent storage + additional cost beyond EC2 instance price

**EMR best practices:**
- Graviton instances = 20% cheaper than x86; use for core + task nodes
- EMRFS (S3) = persistent + decoupled from compute; data survives cluster termination
- HDFS = ephemeral (data lost when cluster terminates); pays for replication
- Never use Spot for core nodes: HDFS data loss risk on Spot interruption

**Redshift distribution styles:**
- **EVEN**: uniform distribution; default; good when no clear join pattern
- **KEY**: collocates matching rows on same slice; optimal for frequent JOIN columns
- **ALL**: entire table copied to every node; eliminates redistribution for joins
- **AUTO**: Redshift decides based on table size; use when no clear pattern

**Redshift ALL distribution for small dimension tables:**
- ALL = every node has a full copy → zero redistribution for any join involving this table
- Only appropriate for small, rarely-updated tables (e.g., dimension/lookup tables)
- ALL for large tables = multiplies storage by node count (expensive, slow inserts/loads)

**Redshift PK/FK constraints:**
- Informational only (not enforced by Redshift); query optimizer uses them to generate better plans
- Define them if application enforces integrity; don't define if application doesn't enforce

**Athena partition optimization for excessive partitions:**
- **Partition Projection**: compute partition values from config instead of querying metastore (eliminates metadata retrieval overhead)
- **Glue Partition Index**: allows `GetPartitions` to fetch a subset instead of loading all partitions
- ORC/Parquet format and compression help query speed but don't fix partition metadata bottleneck

**ElastiCache Redis + DynamoDB DAX for leaderboards:**
- ElastiCache Redis: sub-millisecond in-memory; ideal for gaming leaderboards, real-time rankings
- DynamoDB + DAX: DynamoDB-native in-memory cache; also valid for leaderboard/session use cases
- Neptune = graph DB (not in-memory); Aurora = relational (not in-memory)

**S3 columnar format + partitioning + sorting:**
- ORC or Parquet (columnar): only reads relevant columns → optimal for analytical queries
- Partition by primary filter key (e.g., **date** for daily analysis → scan only 1 day's data)
- Sort by secondary filter key (e.g., **device type** → faster product analysis within partition)
- CSV = row-based = suboptimal for complex analytical queries on large datasets

**S3 Standard → Standard-IA → delete lifecycle:**
- Transition to Standard-IA: minimum 30 days after object creation (lifecycle requirement)
- Standard-IA: low storage cost + per-GB retrieval fee; millisecond access
- Not One Zone-IA for critical/non-reproducible data (single AZ = lower durability)
- Delete at year 5: no need to archive to Glacier if data is not needed after 5 years

**S3 lifecycle for clearly defined access patterns:**
- 0–6 months: S3 Standard; 6 mo – 3 yr: S3 Standard-IA; 3+ yr: S3 Glacier Deep Archive
- Glacier Deep Archive: lowest cost; 12-hour default retrieval; min 180-day storage duration
- When access patterns are clearly known: skip Intelligent-Tiering (unnecessary monitoring cost)

## Domain 2 – Data Store Management (Set 7)

### DMS: Empty Tables Are NOT Migrated
- **Expected behavior** — AWS DMS does not migrate empty tables
- Workaround: insert dummy data into empty tables before the migration task
- Empty tables ≠ lack of resources (slow migration ≠ dropped tables)

---

### ORC vs Parquet — Complex Data Types
- Both are **columnar** storage formats optimized for fast retrieval (compression, predicate pushdown, splitting)
- **ORC**: supports complex types — `structs`, `maps`, `lists`; ORC indexes can make queries faster; use when complex types are required
- **Parquet**: great for nested data in some frameworks; preferred by Spark in many cases
- **Avro**: row-based, schema evolution support, NOT columnar
- **XML**: not optimized for fast retrieval

---

### EBS gp2 → gp3 Migration (Live, No Downtime)
- Use **Modify Volume** in EC2 console → change Volume Type to gp3
- Can simultaneously modify: **size, IOPS, throughput** — no volume size increase required
- **No instance stop, no hibernation** needed — application continues running
- gp3 is ~20% cheaper than gp2

---

### S3 Lifecycle Conflict Resolution (Precedence Rules)
When an object is eligible for multiple Lifecycle actions:

```
Priority 1 (Highest): Permanent DELETE
       │
Priority 2:           TRANSITION to another class
       │
Priority 3 (Lowest):  Create DELETE MARKER

Special case: Glacier Flexible Retrieval wins over Standard-IA or One Zone-IA
```
- S3 objects ARE automatically acted upon — no "none" outcome when multiple rules apply

---

### Athena UNLOAD vs CTAS
| Feature | UNLOAD | CTAS |
|---|---|---|
| Output formats | Parquet, ORC, Avro, JSON | Parquet, ORC, Avro, JSON, CSV |
| Creates new table? | **No** | **Yes** |
| Use when | Need non-CSV without a new table | Need a new table + stored data |
| SELECT command | CSV only | — |

- Use `UNLOAD` when downstream app needs Parquet/JSON output but you don't want a new Athena table

---

### Redshift Spectrum — Query Historical S3 Data
- Query S3 data directly from Redshift **without loading it** into cluster storage
- Create `EXTERNAL TABLE` pointing to S3 path → join with internal Redshift tables
- Compute is handled by Spectrum layer (independent of cluster) — reduces cluster load
- Cost-effective for infrequent historical queries; more efficient than COPY+load for ad-hoc access
- **S3DistCp** is for S3→EMR HDFS only — cannot load data into Redshift

---

### S3 Prefix Performance Scaling
- **3,500 PUT/COPY/POST/DELETE** per second **per prefix**
- **5,500 GET/HEAD** per second **per prefix**
- No limit on number of prefixes
- Solution for high request rates: create **customer-specific prefixes** → parallelize across prefixes

```
Example prefix: s3://bucket/customerA/2024/01/file.csv
                         └─────────────────┘
                              prefix = /customerA/2024/01/
Each prefix gets its own 3,500/5,500 req/sec limit
```

---

### S3 Encryption Options

```
┌─────────────────┬───────────────────┬──────────────┬──────────────────────────┐
│  Type           │  Key Management   │  Audit Trail │  Use Case                │
├─────────────────┼───────────────────┼──────────────┼──────────────────────────┤
│ SSE-S3          │ AWS (automatic)   │     No       │ Default, simplest        │
│ SSE-KMS         │ AWS KMS           │    Yes ✅    │ Compliance / audit req.  │
│ SSE-C           │ You (per-request) │     No       │ Custom / proprietary key │
│ Client-Side Enc │ You (pre-upload)  │     No       │ Proprietary algorithm    │
└─────────────────┴───────────────────┴──────────────┴──────────────────────────┘
```
- **SSE-S3**: each object encrypted with unique key; key encrypted with regularly-rotated root key; **default for all new buckets**
- **SSE-KMS**: AWS manages key lifecycle; CloudTrail logs every CMK use (who, when, what); use when **audit trail needed but don't want to manage keys**
- **SSE-C**: you supply the key per request; AWS encrypts/decrypts; AWS **never stores** your key; use for **proprietary key management**
- **Client-Side**: encrypt before upload; AWS never sees plaintext; use for **proprietary encryption algorithm**

---

### Glacier Vault Lock (WORM Compliance)
- Deploy compliance controls via **vault lock policy** on an S3 Glacier vault
- Specify "write once read many" (WORM) → lock policy from future edits
- Used for **regulatory archival** (healthcare, finance) — different from S3 Object Lock
- ACLs and lifecycle policies cannot enforce compliance controls — only vault lock can

---

### S3 Strong Consistency (since Dec 2020)
- **All operations** are strongly consistent: GET, PUT, LIST, tags, ACLs, metadata
- What you write is what you immediately read — no eventual consistency for any S3 operation
- Simplifies data lake architectures; safe for read-after-write patterns

---

### EC2 Instance Store vs EBS for I/O Performance
| Feature | Instance Store | EBS io1/gp3 |
|---|---|---|
| IOPS | Highest (NVMe SSD) | High (up to 64,000) |
| Latency | Lowest | Low |
| Persistence | Ephemeral (lost on stop/terminate) | Persistent |
| Cost | Included in instance | Additional |
| Use for | Temp scratch, buffers, max I/O | Databases, persistent storage |

---

### S3 Select — Querying Compressed Data Cost-Effectively
- Use SQL clauses (SELECT, WHERE) on CSV/JSON/Parquet stored in S3
- Works with **gzip / bzip2** compressed CSV and JSON
- Much cheaper than Athena or Redshift Spectrum for simple subset queries
- Cannot query Glacier Deep Archive directly — must restore to S3 first
- `ScanRange` parameter enables parallel partial-object scanning

# Domain 3: Data Operations and Support

---

## AWS Glue — ETL Operations

- **Glue Crawler** + **Glue ETL Job** = standard pattern for schema detection + transformation
- For schema-changing sources (undefined/evolving schemas) → **Glue Crawler** detects changes, **Glue ETL (Spark)** processes them
- Use Glue to load incremental data from S3 → Redshift (not Lambda — Lambda 15-min limit)
- Use Glue to convert CSV → Parquet + partition data → dramatically improves Athena performance
- Glue Data Catalog = central metadata store used by Athena, EMR, Redshift Spectrum

### Glue ETL: S3 → Redshift Pattern
1. Third-party exports to S3
2. **AWS Glue ETL job** reads S3, transforms, writes to Redshift
3. Glue is serverless — no infrastructure to manage

---

## AWS Database Migration Service (DMS)

- Continuous replication with HA; source stays **fully operational** during migration
- Streams data to **Redshift and S3**
- During Redshift migration: DMS moves data to S3 first → then loads into Redshift
- DMS replication instance and Redshift cluster must be in the **same region**
- Least development effort vs. Glue (custom ETL scripts) or EMR (cluster setup)

---

## Amazon EMR

### Library Installation (Automation)
Two correct automated approaches:
1. **Bootstrap actions** — scripts in S3, run before cluster starts, before EMR apps install
2. **Custom AMI** — EC2 + Amazon Linux + install libraries → create AMI → launch EMR from that AMI

### EMR File Systems
- **HDFS** = Hadoop Distributed File System (local to cluster; lost on termination)
- **EMRFS** = implementation of HDFS that reads/writes to S3 using Hadoop FileSystem API — object store, NOT a file system replacement
- **Cannot configure EMR to use S3 as the Hadoop storage layer** (EMRFS ≠ HDFS replacement)
- Amazon S3 block file system is **deprecated** — use EMRFS

### EMR Auto Scaling
- Use **Instance Groups** (not EC2 Spot Fleets — Spot Fleets are EC2-level, not EMR-level)
- Scale metric: **`YARNMemoryAvailablePercentage`** = remaining YARN memory %
- `CapacityRemainingGB` = HDFS disk space — useful for monitoring but NOT for scaling (HDFS often under-used)

### EMR — Apache Hive
- Natively supported on EMR
- Distributed, fault-tolerant data warehouse query system

---

## Amazon CloudWatch + CloudTrail

### Monitoring API Calls
- CloudTrail logs → CloudWatch Logs → **Metric Filter** → CloudWatch Alarm → **SNS notification**
- CloudTrail **cannot** stream directly to Kinesis (only S3 and CloudWatch Logs are valid destinations)
- **CloudTrail Insights** — detects unusual write API activity automatically

### EC2 Auto Recovery with CloudWatch Alarms
- Recovered instance is **identical**: same instance ID, private IPs, EIPs, metadata
- **Retains public IPv4** after recovery
- **Terminated instances cannot be recovered**
- During recovery: instance reboots and **in-memory data is LOST**

---

## Amazon Kinesis — Operations

### Resharding
- **Split** → more shards → higher throughput → higher cost
- Triggered by `ProvisionedThroughputExceededException`
- Long-running high lag → increase shards to reduce `GetRecords.IteratorAgeMilliseconds`

### KCL / KPL Duplicate Records
- Producer retry after network timeout → same record written twice
- Embed **primary key** in record payload to deduplicate downstream

### Kinesis Metrics to Know
| Metric | Measures |
|--------|----------|
| `GetRecords.IteratorAgeMilliseconds` | Age of last record; 0 = real-time |
| `GetRecords.Latency` | Time per GetRecords call |
| `PutRecords.Latency` | Time per PutRecords call |
| `ReadProvisionedThroughputExceeded` | GetRecords throttle count |

---

## AWS Step Functions + Long-Running Athena Queries

**Pattern for scheduling Athena queries (> 15 min):**
1. **Lambda** calls `start_query_execution` (Athena Boto3 API) — cheapest trigger
2. **Step Functions** state machine:
   - State 1: invoke Lambda
   - State 2: **Wait state** → poll `get_query_execution` until complete
   - State 3: trigger next query

> Using AWS Glue Python shell instead of Lambda is valid but **more expensive**.  
> Using a sleep timer in a Glue job is resource-inefficient.

---

## DynamoDB Streams + Lambda

- Streams capture ordered item-level changes for **24 hours**
- Lambda triggered on each change event — minimal latency
- Use case: notifications, event-driven workflows triggered by DB changes
- SQS + Lambda also works but requires extra code to write to SQS first

---

## Amazon ECS with Application Load Balancer

- If 90% of traffic is static content → offload to **S3** (not EFS — EFS still requires EC2 to serve)
- EFS is a network file system; it does not serve HTTP requests directly

---

## Networking: Security Groups vs NACLs

| Feature | Security Group | Network ACL (NACL) |
|---------|---------------|---------------------|
| **Stateful** | ✅ Yes — return traffic auto-allowed | ❌ No — must explicitly allow both directions |
| Level | Instance (ENI) | Subnet |
| Rules | Allow only | Allow and Deny |
| Outbound for clients | Auto-allowed (stateful) | Must allow **ephemeral ports (1024–65535)** outbound |

> Cannot connect to EC2 service despite correct SG rules? → Check **NACL outbound ephemeral ports**.

---

## Monitoring Streaming Lag — Real-Time Alerts

- For near-real-time API abuse detection: **CloudWatch Metric Filter on CloudTrail logs** → Alarm → SNS
- Athena + QuickSight = reporting (not real-time alerting)
- Trusted Advisor alarms only trigger on **service limit reached** (not anomaly detection)


## Domain 3 — Additional Topics (Set 3)

### Athena Query Result Reuse
- Enable **query result reuse** for dashboards where many users run the same queries
- Results are **shared within a workgroup**; specify maximum age of results to reuse
- Reduces cost significantly when source data doesn't change frequently
- Best when: dashboards + multiple users + daily data refresh (all users avoid re-scanning same data)

### Athena Best Practices
- Use S3 bucket in **same Region as Athena** for best performance and lowest cost
- Athena is serverless; no infrastructure to manage; pay only per query scanned
- Use for: data sanity checks on S3 data lake, ad-hoc analysis, interactive queries

### S3 Glacier Deep Archive
- Lowest-cost storage option in AWS (~$1/TB-month)
- **Standard retrieval: 12 hours; Bulk retrieval: up to 48 hours**
- Use for: compliance data retained 5-10 years accessed once or twice a year
- Glacier Flexible Retrieval = 1-5 min expedited or 5-12 hr bulk; higher cost than Deep Archive

### Redshift Serverless
- Automatically provisions capacity; scales in seconds; pay-per-use
- No cluster setup, tuning, or maintenance
- Best for: monthly analytics runs; eliminate EventBridge + Step Functions complexity for scheduling

### QuickSight as Direct Data Source Connector
- QuickSight connects **directly** to: Redshift, S3, RDS, Aurora, Athena, Salesforce, SQL Server, MySQL, PostgreSQL, Excel, CSV
- No need to migrate all data to S3 or build ETL pipelines for a simple dashboard
- Least effort = configure QuickSight to connect to existing data sources directly

### Lambda Maximum Timeout
- Lambda max execution time = **15 minutes** — not suitable for long-running batch jobs
- If a job is expected to run 30 minutes → use Glue ETL job or EMR instead
- Lambda timeout causes abrupt function termination at exactly 15 min

### Firehose Small Files Root Cause
- When Firehose **auto-scales** (delivery stream capacity increases), buffer size is halved proportionally
- 2x scale → buffer halves → 2 parallel channels → 2x more files in same interval
- 4x scale → buffer quarters → 4 parallel channels → 4x more files
- This is expected behavior; not a misconfiguration

### Glue Job Bookmarks Fix
- Problem: Glue job reprocesses already-processed data despite bookmarks being enabled
- Fix 1: Add `job.commit()` at end of the script — records the timestamp and path of the last run
- Fix 2: Set **max concurrency = 1** for the job — concurrent jobs with bookmarks cause incorrect behavior
- Note: transformation context must be included in DynamicFrame for bookmarks to work (not a bug)

### DMS Replication Instance Placement
- Place replication instance in the **same AZ and VPC as the TARGET DB instance** for optimal performance
- Reduces latency between replication instance and destination
- Cross-region migration: create replication instance in the destination Region's VPC

### EMR High Availability
- Multi-master requires **3 master nodes** in a **single Availability Zone** (NOT multiple AZs)
- Use **EMRFS** (S3-backed) for persistent storage — HDFS is ephemeral (lost when cluster terminates)
- Use **Glue Data Catalog** as Hive metastore — MySQL on master node is lost when cluster terminates
- EMR cluster can only reside in a single Availability Zone

### Step Functions Map State vs Parallel State
- **Map state**: processes each **item in a dataset** in parallel
  - Inline mode: up to **40 concurrent** iterations; accepts JSON array
  - Distributed mode: up to **10,000 concurrent** child workflow executions; supports S3/CSV input
- **Parallel state**: executes fixed predefined **branches** concurrently; waits for all branches to finish
- Use Map for dynamic dataset processing; use Parallel for fixed workflow branches

### DynamoDB Streams → KDA Pipeline
- DynamoDB Streams captures item-level changes for up to **24 hours**
- Pattern: DynamoDB Streams → Lambda → **KDS → KDA** → SNS for anomaly detection
- **KDA only accepts KDS or Firehose as streaming sources** — cannot write Lambda output directly to KDA
- CloudTrail captures DynamoDB API calls but **not** stream record data (GetRecords not supported in CloudTrail)

### RDS Connection Timeout Troubleshooting
- "Connection timed out" = **security group** issue, not credentials issue
- Check inbound rules on the **RDS security group** — must allow traffic from the application server SG
- Bastion host can connect ≠ app server can connect (different security group rules)
- SGs are stateful; no need to add outbound rules for return traffic

### RDS Monitoring: Performance Insights vs Enhanced Monitoring
- **Performance Insights**: identifies slow SQL queries; visualizes DB load by waits/SQL/hosts/users
- **Enhanced Monitoring**: OS-level metrics (CPU per process/thread, memory, disk); stored in CloudWatch Logs
- Use both together: PI for SQL-level root cause; Enhanced for OS-level details
- CloudWatch standard metrics = hypervisor-level only (less granular than Enhanced Monitoring)

### AWS DataSync for Scheduled On-Prem Migration
- DataSync = secure, automated, scheduled data transfer from on-premises → S3/EFS/FSx
- Best for: recurring scheduled data sync with ~2% daily changes
- Not S3 Transfer Acceleration (optimizes transfers, not migration); not Glue (complex setup for on-prem)


## Domain 3 – Set 5: Data Operations and Support

**Athena EXPLAIN vs EXPLAIN ANALYZE:**
- `EXPLAIN`: shows distributed execution plan only (analyze complexity, validate SQL syntax before running)
- `EXPLAIN ANALYZE`: execution plan **+** computational cost per operation **+** scanned data count → financial impact analysis
- PREPARE = parameterized SQL for later execution; OPTIMIZE = Iceberg table optimization

**KDS Enhanced Fan-Out:**
- Default: 2 MB/sec/shard **shared** across all consumers (read via GetRecords polling)
- Enhanced Fan-Out: each registered consumer gets **dedicated** 2 MB/sec/shard (push via SubscribeToShard)
- Use when multiple consumers read same stream in parallel and need independent throughput

**CloudTrail data events for S3 write logging (least effort):**
- Create trail → log **Write** data events on Orders bucket with **empty prefix** (captures ALL objects)
- Using "All-objects" as a prefix = literal string, not a wildcard → wrong
- S3 Event + Lambda = extra complexity; EventBridge + Lambda = unnecessary overhead

**SQS as buffer for DynamoDB write spikes:**
- SQS Standard Queue: infinitely scalable, cost-effective, decouples app layer from DB
- DAX = read cache only (not write buffer); KDS = less cost-effective for write spikes
- SNS = no message persistence (lost if not delivered)

**EventBridge cron + Lambda for serverless scheduled jobs:**
- Pattern: EventBridge cron expression → Lambda (Python/Node/etc.) runs the task
- Not EC2 Spot (no guaranteed execution + not serverless); not EC2 On-Demand (not serverless)
- Not Glue job (overkill for simple scripts, costlier than Lambda); Lambda = pay only for execution time

**DynamoDB RCU formula:**
- 1 RCU = 1 strongly consistent read/sec for items up to 4 KB
- Eventually consistent: 2 reads/sec per RCU (half cost); Transactional: 0.5 reads/sec per RCU (2× cost)
- Formula (strongly consistent): `ceil(item_KB ÷ 4) × reads_per_sec = RCUs`
  - 20 RCU → 80 KB/sec strongly consistent; 160 KB/sec eventually; 40 KB/sec transactional
  - 10 reads/sec × 8 KB items → (8÷4)×10 = **20 RCU**

**Step Functions + EMR failure troubleshooting:**
- Check 1: IAM permissions — Step Functions needs EMR + S3 access; use **S3 Access Analyzer** to verify S3 bucket permissions
- Check 2: VPC flow logs — verify EMR cluster traffic can reach data sources; check security group inbound rules/ports
- Fail state = captures failure but does not identify root cause

**S3 Select for single-object column queries:**
- SQL `SELECT` on a **single S3 object** (CSV, JSON, Parquet); supports GZIP/BZIP2 compression
- Only supports `SELECT`; one object at a time; least operational overhead for ad-hoc single-file queries
- vs Athena: Athena queries multiple objects/tables across S3 (more setup with Glue crawler)

**Lake Formation for row + column access control:**
- Provides fine-grained access at column, row, and cell levels via data filters
- Works with: Athena, Redshift Spectrum, EMR, AWS Glue, QuickSight
- S3 Access Points: cannot do row-level filtering; IAM: cannot do row-level for S3 objects
- Apache Ranger via EMR: also supports column/row-level but much higher operational overhead

**EBS-backed AMI + additional EBS volume for data persistence:**
- Root EBS volume: **deleted on termination by default** (Delete on termination = true by default)
- Additional EBS data volumes attached at launch: **persist after termination by default**
- Instance store-backed AMI: root = ephemeral (lost on termination); cannot attach instance store as additional data volume to EBS-rooted instance
- Solution: instance store AMI + attach additional EBS volume for application data

**Redshift vacuum for post-nightly-refresh performance degradation:**
- Vacuum: re-sorts rows + reclaims space freed by DELETE/UPDATE operations
- Auto vacuum pauses when cluster has high query load (nightly refresh spikes prevent vacuum from completing)
- Result: tables remain unsorted/fragmented → performance degradation persists during business hours
- CloudWatch shows no alarms because it's a data quality issue, not a resource limit issue

**Redshift distribution key change for load skew:**
- Symptom: one node at 90% CPU, others at 10% = wrong distribution key (skewed data)
- Fix: change DISTKEY to column with **largest dimension** (highest cardinality, most distinct values)
- DISTKEY = column used in JOINs; SORTKEY = column used in WHERE clauses (different purposes)
- Changing to ALL distribution = multiplies storage × node count (not feasible without adding nodes)

**Athena partitioning strategy:**
- Partition by the column most commonly used as a query filter to reduce data scanned
- Example: pet shop daily analysis → partition by **pet type** (not customer, not store region)
- Wrong partition key = full table scan instead of partition pruning

**DynamoDB bulk delete: delete + recreate table:**
- DeleteTable + CreateTable = most cost-efficient for deleting ALL items in a large table
- Scan + BatchWriteItem (delete mode) = extremely slow and expensive for millions of items
- "Purge table" option = does not exist (distractor)

**Redshift monitoring system tables/views:**
- `STL_ALERT_EVENT_LOG`: logs inefficient/long-running queries with performance improvement recommendations
- `SVV_TRANSACTIONS`: shows current transactions holding locks on tables (lock contention)
- `STL_QUERY_METRICS`: completed query metrics (CPU, I/O, rows); `STL_SESSIONS`: long-running sessions

**Lambda provisioned concurrency for API Gateway (no cold starts):**
- Provisioned concurrency: pre-initializes execution environments → responds immediately on first request
- Use for API Gateway backends requiring consistent low latency (e.g., real-time REST APIs)
- **EventBridge Scheduler pings do NOT keep Lambda warm** → this is a common misconception; use provisioned concurrency instead

**DynamoDB on-demand vs provisioned:**
- On-demand: instant scaling to any traffic level; pay-per-request; no capacity planning required
- Provisioned + auto-scaling: CloudWatch alarms → Application Auto Scaling (delay in scaling); good only for predictable/gradual traffic
- Use on-demand when: new tables, unpredictable/bursty traffic, spikes occur in seconds

**Redshift ALL distribution for small tables + PK/FK constraints:**
- ALL distribution: every node has full copy → zero redistribution cost for joins with this table
- Appropriate only for small (< ~1 GB) and rarely-updated dimension/lookup tables
- PK/FK constraints: informational only (not enforced), but optimizer uses for better query plans

**DAX for DynamoDB + CloudFront for S3:**
- DAX: DynamoDB-native in-memory cache; API-compatible (no code changes); up to 10× read improvement
- CloudFront: CDN for S3 static content; caches at edge → reduces S3 outbound cost + latency
- ElastiCache: cannot natively cache DynamoDB (requires custom logic); cannot cache S3 content

**Glue crawler exclude pattern for slow crawls:**
- Use **exclude patterns** to skip unwanted files (e.g., data older than 1 week)
- Splitting files into smaller = slower (crawler must list + read each file individually)
- Scanning rate parameter = only available for DynamoDB stores (not S3)
- Parquet/ORC/Avro: crawler reads metadata (faster than raw row-based files)

## Domain 3 – Data Operations and Support (Set 7)

### Glue Workflows vs Step Functions
| Orchestrator | Best For | Supports Lambda? |
|---|---|---|
| **Glue Workflows** | Glue-only pipelines (crawlers + Spark + Python shell jobs) | ❌ No |
| **Step Functions** | Mixed pipelines (Lambda + Glue + other AWS services) | ✅ Yes |
| Amazon MWAA | Complex multi-tool workflows (high management overhead) | ✅ Yes |

- Glue Workflow = multi-job + multi-crawler graph; run on demand or schedule
- If the pipeline includes **Lambda functions**, use **Step Functions** (Glue Workflow cannot include Lambda)

---

### DynamoDB Application Auto Scaling
- Use **provisioned mode + Application Auto Scaling** for **predictable demand with spikes**
- Set min/max capacity + target utilization % → CloudWatch alarms trigger scaling
- Takes **several minutes** to react — not instant (burst capacity handles brief spikes)
- On-demand = instant scaling but costs more for predictable workloads
- GSI/LSI improve query access patterns but do **not** address traffic spikes

---

### QuickSight → Redshift Connectivity
- Error "connection failing" = **security group not configured**
- Fix: add QuickSight IP CIDR range for the **AWS Region** as inbound rule on Redshift security group
- If QuickSight active in multiple regions → add inbound rules for each region's CIDR block
- NOT IAM role (wrong error type) — IAM issues produce `UnauthorizedOperation` / `AccessDenied` errors

---

### KCL Shard Load Balancing

```
Stream: [Shard-1][Shard-2][Shard-3][Shard-4]

1 EC2 instance:
  ┌────────────┐
  │ Instance A │ → processes all 4 shards (4 record processors)
  └────────────┘

2 EC2 instances (KCL auto load-balances):
  ┌────────────┐       ┌────────────┐
  │ Instance A │       │ Instance B │
  │  Shard 1   │       │  Shard 3   │
  │  Shard 2   │       │  Shard 4   │
  └────────────┘       └────────────┘

Rule: instances should NOT exceed number of shards
      (extra instances = standby/failover only)
```

- KCL automatically discovers new shards after resharding and redistributes
- One shard = exactly one KCL worker; one worker can process many shards

---

### KCL DynamoDB Requirement
- Each KCL application **must have its own DynamoDB table** (for lease management + checkpointing)
- Cannot share tables across KCL apps → would cause lease/shard ID conflicts
- **Only DynamoDB** supported for checkpointing (not RDS or S3)
- KCL uses DynamoDB for: ShardSyncTask, DynamoDBLeaseTaker, DynamoDBLeaseRenewer, checkpointing

---

### Athena Cost Controls

```
Per-Query Limit:
  ┌─────────────────────────────────────────────────────┐
  │  1 limit per workgroup                              │
  │  → query CANCELLED if it exceeds the data limit    │
  └─────────────────────────────────────────────────────┘

Per-Workgroup Limit:
  ┌─────────────────────────────────────────────────────┐
  │  N limits per workgroup (multiple thresholds)       │
  │  Hourly aggregate: SNS alert at X GB scanned        │
  │  Daily aggregate:  SNS alert at Y GB scanned        │
  │  → query NOT cancelled; only SNS notification sent  │
  └─────────────────────────────────────────────────────┘
```
- Use **per-workgroup limits** when you need hourly/daily aggregate thresholds

---

### S3 Byte Range Fetch + S3 Select ScanRange
- **Byte Range Fetch**: `GET` object with `Range: bytes=0-249` HTTP header → fetch only first 250 bytes
  - Avoid reading 50 TB to build an index; parallel fetching of different byte ranges
- **S3 Select ScanRange**: specify `ScanRange` (start byte, end byte) within an object's SQL scan
  - Enables parallel S3 Select across non-overlapping ranges of a single object
- Both avoid reading entire large files — critical for cost and performance at scale

---

### Glue FindMatches ML Transform
- Built-in ML for **record deduplication** — no code required
- Teach by labeling example pairs (match / no-match)
- Identifies duplicates even without a common unique identifier
- Superior to `DataFrame.dropDuplicates()` (which requires exact field matches) for fuzzy deduplication

---

### Glue + Secrets Manager Integration
- **Never** store passwords in Glue job parameters (can be read by anyone with job access)
- Store credentials in **AWS Secrets Manager** → attach IAM role with Secrets Manager permissions to Glue job
- AWS best practice; credentials accessed at runtime, never visible in scripts

---

### DMS Data Validation
- Compares **source vs target records** row by row after migration and reports mismatches
- Works for full load + CDC (ongoing replication)
- Consumes additional resources on source and target during comparison
- Different from **premigration assessment** (identifies potential migration blockers before running)

---

### Athena Spark Workgroup
- To use Apache Spark on Athena: **create a Spark-enabled workgroup** first
- After switching to the workgroup, create/open a notebook
- Python-based; serverless; no cluster management needed
- NOT: update Athena engine version, use query editor toggle, or any settings panel

---

### Glue FLEX Execution Class
- For **non-urgent, non-time-sensitive** jobs: testing, pre-prod, one-time data loads
- Runs on **spare compute capacity** → cheaper than STANDARD class
- Start and run times may vary; capacity can be reclaimed mid-job
- **STANDARD**: dedicated resources, fast startup — use for time-sensitive production jobs
- Billing: by number of workers running at any point in time

---

### EFS Performance Modes
| Mode | Best For | Tradeoff |
|---|---|---|
| **Max I/O** | Big data, analytics, media processing, genomic analysis | Slightly higher metadata latency |
| **General Purpose** | Web serving, CMS, home dirs (default) | Lower latency, lower aggregate throughput |

- Throughput modes (separate from performance modes): Bursting vs Provisioned

---

### Kinesis Firehose → OpenSearch (Kibana Dashboards)
- Firehose has **native built-in support** for OpenSearch Service as a destination
- OpenSearch (Kibana) dashboards support **auto-refresh** — suitable for near-real-time analytics
- QuickSight refresh minimum = hourly (Enterprise) → not suitable for 10-second refresh

---

### SQS for Microservices Decoupling
- SQS stores messages until downstream consumer is ready → decouples fast producers from slow consumers
- SNS = push-based (assumes consumer is always available) → tightly coupled for async workflows
- EventBridge = event-based (downstream must process immediately) → not for buffering
- KDS = streaming (not designed for microservice decoupling with variable-speed consumers)

---

### DynamoDB Strongly Consistent Reads
- `ConsistentRead = true` applies only to **read operations**: `GetItem`, `Query`, `Scan`
- Cannot set ConsistentRead on `PutItem` or `UpdateItem` (write operations don't have this parameter)
- Default = eventually consistent reads; use ConsistentRead=true to always get latest written value

# Domain 4: Data Security and Governance

---

## EC2 Instance Recovery (CloudWatch Alarm)

✅ After recovery, instance retains:
- Same **instance ID**
- Same **private IP addresses**
- Same **Elastic IP addresses (EIPs)**
- Same **public IPv4 address**
- All **instance metadata**

❌ These do NOT apply:
- **Terminated instances cannot be recovered** (only impaired/failed instances)
- **In-memory data is LOST** (instance reboots during recovery)

---

## Amazon EBS Encryption

When you encrypt an EBS volume, **all of the following are encrypted automatically:**
- ✅ Data **at rest** inside the volume
- ✅ Data **in transit** between volume and instance
- ✅ **Snapshots** created from the volume
- ✅ Volumes created from encrypted snapshots

> Encryption uses **AWS KMS CMKs**.  
> Operations occur on the EC2 host — transparent to the OS.

---

## Amazon S3 Encryption

| Type | Who manages keys | Algorithm |
|------|-----------------|-----------|
| **SSE-S3** | AWS manages everything | AES-256 |
| **SSE-KMS** | AWS KMS (audit trail, key policies) | AES-256 + KMS key |
| **SSE-C** | Customer provides key each request | Customer's key |
| **Client-Side** | Customer encrypts before upload | Customer's choice |

> **Metadata is NOT encrypted** by any server-side encryption method — never store sensitive data in S3 object metadata.

> Exam asks for AES-256 + AWS manages keys + no extra cost → **SSE-S3** (default since 2023).  
> SSE-KMS has additional KMS usage fees.

### Data In Transit
- S3 supports **HTTPS (TLS)** for encryption in transit
- You can enforce HTTPS via bucket policy with `aws:SecureTransport: false → Deny`

---

## AWS Certificate Manager (ACM)

- **Fully managed** provisioning, renewal, and deployment of SSL/TLS certificates
- Certificates auto-renew before expiration
- Attach to: **ELB, CloudFront, API Gateway**
- Supports public and private CAs (ACM Private CA)
- Zero cost for ACM-issued certificates (only pay for private CA if used)
- Eliminates manual cert management, cron jobs, shell scripts

---

## AWS Secrets Manager

- Store and **automatically rotate** database credentials, API keys
- Built-in rotation integration for: **RDS, Redshift, DocumentDB**
- 90-day rotation example: set rotation interval in Secrets Manager — no custom code needed
- Applications call Secrets Manager API at runtime — no hardcoded secrets

> SSM Parameter Store stores secrets but does **NOT** natively rotate credentials.  
> Writing custom rotation scripts (Lambda, cron, S3 files) = incorrect for "built-in integration" requirement.

---

## Security Groups (Stateful) vs NACLs (Stateless)

| | Security Group | NACL |
|--|---------------|------|
| **Type** | Stateful | Stateless |
| Return traffic | Auto-allowed | Must explicitly allow |
| Direction rules needed | Inbound only (return is automatic) | Both inbound **AND** outbound |
| Ephemeral ports | N/A | Must allow outbound **1024–65535** |

> **Common exam trap:** Configured SG inbound + NACL inbound but still can't connect → NACL is missing **outbound** ephemeral port rule.

---

## AWS Glue Crawler — Schema Detection Rules

- Crawler compares schemas across files in a folder
- If **> 70% of files** share same schema (partition threshold) → **1 merged table**
- If **≤ 70%** share same schema → **separate tables** per schema
- Maximum **5 clusters** (different schemas) per folder before separate tables are forced

| Folder | Files same schema | % | Result |
|--------|------------------|---|--------|
| FOLDER1 | 8/10 SCH_A | 80% | ✅ > 70% → 1 merged table |
| FOLDER2 | 7/10 SCH_A | 70% | ❌ = 70% (not >) → 2 separate tables |

---

## EMR + S3: EMRFS vs HDFS

- **EMRFS** = reads/writes from EMR to S3 using Hadoop FileSystem API; provides encryption, persistence
- **HDFS** = runs on EMR cluster nodes; local, fast, ephemeral
- **You CANNOT replace HDFS with S3** as the Hadoop storage layer — they are not interchangeable
- EMRFS is an **object store**, not a POSIX file system
- S3 Block File System = **deprecated** (used before multipart upload); avoid it

---

## Athena Workgroups — Team Isolation & Cost Control

- Create one workgroup per team
- Apply **tags** → use tags in IAM policy for permission control
- Each workgroup can have a **data scan limit** (cost control)
- Query history is **isolated per workgroup** — other teams cannot see it
- Lake Formation does NOT control Athena query history — workgroups do
- IAM groups alone cannot restrict access to Athena query history

---

## VPC Gateway Endpoint for S3 + AWS Glue

- Gateway endpoint = private connection from VPC to S3 (no internet, no NAT, no IGW needed)
- Endpoint adds a **route** to the VPC route table automatically
- If Glue job fails pointing to VPC gateway endpoint → **verify route table has inbound and outbound routes** for the endpoint prefix list
- Private DNS options apply to **interface endpoints** (not gateway endpoints)

---

## AWS Data Exchange

- Subscribe to and receive **third-party datasets** directly in AWS
- Data sets available via AWS Marketplace catalog
- No custom integration needed — data directly accessible in S3/analytics services
- Lower overhead than DataSync (migration tool) or CodeCommit (source control — wrong service entirely)

---

## Quick Security Decision Tree

```
Need to encrypt S3 data at rest, AWS manages keys, AES-256, no extra cost?
→ SSE-S3

Need key audit trail / key policies / willing to pay KMS fees?
→ SSE-KMS

Need to bring your own key each request?
→ SSE-C

Need to auto-rotate DB credentials with built-in AWS integration?
→ Secrets Manager

Need SSL/TLS certs auto-renewed, attached to ELB/CloudFront?
→ ACM

Need to query S3 privately from Glue without internet?
→ VPC Gateway Endpoint for S3 (check route table!)

Need to isolate Athena usage/costs between teams?
→ Workgroups + IAM tag-based policies
```


## Domain 4 — Additional Topics (Set 3)

### Site-to-Site VPN vs Direct Connect
- **Site-to-Site VPN**: fastest to set up; encrypted via **IPSec**; uses public internet; variable performance
- **Direct Connect**: dedicated private connection; **no encryption in transit by default**; consistent performance; costly
- Pattern for hybrid: **Direct Connect = primary** (performance), **VPN = backup** (encrypted failover)
- Internet Gateway ≠ network connectivity to on-premises; DataSync ≠ network connectivity

### Glue Data Quality + DQDL
- **DQDL** (Data Quality Definition Language) = declarative rules like `IsComplete "transaction_id"`, `RowCount > 0`
- Add a Data Quality node in Glue Studio → conditionally route records: pass → write to S3; fail → error path
- Least effort vs Step Functions + Lambda (adds complexity) or imputing values (changes data)
- Supports row-level and dataset-level rules; built into existing Glue pipeline with no new services

### RDS Encrypted Snapshot Cross-Account Sharing
- **Cannot share** a snapshot encrypted with the **default AWS KMS key**
- Steps: (1) Add target account to a **customer-managed KMS key**, (2) copy snapshot using CMK, (3) share the copy, (4) target account copies the shared snapshot
- Default KMS key cannot be shared; all three steps with CMK are required

### Redshift Column-Level Access
- `GRANT SELECT (col1, col2) ON TABLE t TO GROUP finance` — column-level GRANT supported since 2020
- No need for views or duplicate tables — adds unnecessary storage and maintenance
- Do NOT grant admin to the finance team — violates least privilege

### QuickSight Editions
- **Standard**: no Active Directory integration, no encryption at rest
- **Enterprise**: AD integration, encryption at rest, SAML 2.0 federation for SSO
- **All editions**: keys managed by AWS — **no customer-managed KMS keys** for QuickSight
- Use SAML 2.0 (not AD Connector) for AD federation in QuickSight Enterprise

### S3 Object Deletion Protection
- Enable **S3 Versioning**: deletes create a delete marker (object not permanently removed); previous versions recoverable
- Enable **MFA Delete**: requires MFA device to permanently delete objects — extra protection layer
- S3 event notification after delete = too late (object already gone); no console confirmation config exists

### CloudFront + WAF + OAC for IP Restriction
- **OAC (Origin Access Control)**: restricts S3 bucket access to CloudFront only (bucket not publicly accessible)
- **WAF ACL with IP match condition**: restrict CloudFront access to specific IP ranges
- Associate WAF ACL with **CloudFront distribution** — not with S3 bucket policy
- NACL = subnet-level only; SG = EC2-level only — **neither can be associated with CloudFront**

### AWS WAF Deployment Targets
- WAF can be deployed on: **CloudFront, ALB, API Gateway, AppSync, Cognito user pools**
- **Cannot** be deployed directly on EC2 instances
- To protect EC2 app: create **CloudFront distribution → deploy WAF on CloudFront**

### Lake Formation for Data Mesh
- Lake Formation = centralized governance + fine-grained access control for data lakes on S3
- **Tag-Based Access Control (TBAC)**: attach LF-tags to databases/tables/columns → manage permissions via tags (not named resources)
- Scales permissions management; decouples policy creation from resource creation
- Pattern: **S3 (storage) + Glue Data Catalog (catalog) + Athena (query) + Lake Formation (governance)**

### Macie vs DataBrew for PII
- **Macie**: discovers and **classifies** PII in S3 using ML — detection only, **no masking**
- **DataBrew**: profile job identifies PII columns → apply masking/redaction/hashing transformations
- **Glue Detect PII transform**: identifies and masks PII natively in Glue Studio pipeline
- **Glue Data Quality**: validates data rules (completeness, accuracy) — does **NOT** mask PII
- Combined approach: Macie (find) → DataBrew or Glue (mask)

### IAM for Sensitive S3 Data
- Define granular IAM policies restricting access to sensitive S3 prefixes/objects by role
- Can add conditions: VPC endpoint, MFA, encryption context
- Principle of least privilege: analysts get masked bucket; engineers get unmasked bucket (role-based)

### Security Group Design Pattern (ALB → EC2 → RDS)
- **ALB SG**: inbound port **443** from `0.0.0.0/0` (HTTPS from internet)
- **EC2 SG**: inbound port **80** from ALB SG (HTTP from ALB only)
- **RDS SG**: inbound port **5432** (PostgreSQL) from EC2 SG
- SGs are **stateful** — return traffic automatically allowed; no outbound rules needed for responses

### AWS Config vs CloudTrail vs CloudWatch
- **AWS Config**: resource configuration history + compliance; "What did my resource look like at time X?"
- **CloudTrail**: API call audit; "Who made this API call?"
- **CloudWatch**: performance metrics, events, alarms; "Is my resource performing normally?"
- Use Config for compliance audits and configuration change history

### Redshift Cross-Region Snapshots (KMS)
- Create **snapshot copy grant in DESTINATION region** for a KMS key **in DESTINATION region**
- Configure **cross-region snapshots in SOURCE region** — not destination region
- AWS KMS keys are region-specific — cannot reference source region key from destination
- Cross-region replication = S3 concept; Redshift uses cross-region snapshot copy

### CloudTrail for S3 Audit
- Use CloudTrail to analyze **API calls made to S3** (who changed bucket settings, when, how)
- Recommended over S3 access logs for: more options to store, analyze, and act on logs
- CloudTrail logs to S3 with continuous delivery; use Athena to query CloudTrail logs

### SSE-C (Server-Side Encryption with Customer-Provided Keys)
- Customer **provides and manages** the encryption key; AWS handles encryption/decryption
- Uses **AES-256** algorithm; key is included in each request header
- Use when: company must manage its own encryption keys but wants AWS to do the encryption
- SSE-KMS = AWS manages keys (never see raw key); SSE-S3 = fully AWS-managed; Client-side = encrypt before upload

### QuickSight → Redshift Cross-Region Access
- QuickSight in Region A connecting to Redshift in Region B requires:
  - Add **inbound rule to Redshift SG** authorizing the QuickSight CIDR block for the QuickSight region
  - Works whether Redshift is in VPC or not
- VPC endpoint = only within the same region; Redshift snapshot copy = expensive and not needed

### Snow Family — Clustering Feature
- **Snowball Edge Storage Optimized** and **Snowball Edge Compute Optimized** both support clustering
- **Snowcone**: no clustering (too small)
- **Snowmobile**: no clustering (single container unit)
- Clustered Snowball Edge devices can be rack-mounted and used as temporary installations


## Domain 4 – Set 5: Data Security and Governance

**AWS Secrets Manager for DB credential auto-rotation:**
- Rotates credentials automatically for RDS, Redshift, DocumentDB; eliminates hardcoded connection strings
- SSM Parameter Store: no auto-rotation; AWS Config: compliance auditing only; KMS: key management only
- Secrets Manager API call retrieves secrets at runtime (no hardcoding)

**S3 data lake lifecycle: raw zone compliance + curated zone analytics:**
- Raw zone (source data, 5-year compliance): lifecycle to **Glacier Deep Archive after 1 day** (cannot delete)
- Curated zone (ad-hoc queries): store in **compressed columnar format** (Parquet/ORC) — cannot archive to Glacier (actively queried by Athena)
- Never delete raw zone data (compliance violation); never move curated to Glacier (breaks ad-hoc queries)

**Step Functions for serverless orchestration:**
- Coordinates Lambda, Glue, EMR, S3, SNS with built-in retry/error handling; fully serverless
- Glue workflows: cannot invoke Lambda or other non-Glue services → wrong for generic orchestration
- MWAA (Apache Airflow): higher operational overhead (needs configuration, setup, managed environment)

**WAF IP match condition to block malicious IPs:**
- Create IP match condition → allow/block up to 10,000 IP addresses or ranges
- Attach WAF Web ACL to CloudFront, ALB, or API Gateway (**cannot attach WAF directly to EC2**)
- Security Groups: cannot create DENY rules (allow-only); NACLs: subnet-level (not instance-level)
- Managing application security = your responsibility (cannot raise support ticket for this)

**VPC Gateway Endpoints — only S3 and DynamoDB:**
- Gateway Endpoints: target in route table; traffic stays on AWS network; no ENI created
- Supports: **Amazon S3** and **Amazon DynamoDB only**
- All other services (SQS, SNS, Kinesis, ECR, etc.): use **Interface Endpoints** (ENI with private IP via PrivateLink)

**CloudFront + S3 for outbound cost reduction:**
- No data transfer fee from S3 to CloudFront; only pay for CloudFront → internet delivery
- CloudFront caches at edge → reduces S3 direct outbound requests (much cheaper at scale)
- EBS = only accessible via EC2 (not scalable/suitable for web content); EFS = costlier than S3
- S3 Batch Operations = for bulk object operations (copy, tag, invoke Lambda) — not content delivery

**S3 static website + CloudFront for HTTPS:**
- S3 website endpoints do **not** support HTTPS natively
- Solution: CloudFront distribution with S3 as origin → HTTPS + low latency + custom domain
- Fully serverless; no EC2 or container infrastructure required

**Client-Side Encryption for proprietary algorithm:**
- Data encrypted **before** reaching AWS → AWS never sees plaintext; proprietary algorithm runs locally
- SSE-C: AWS encrypts using your provided key (AWS handles encryption process)
- SSE-KMS / SSE-S3: AWS fully manages encryption (no room for proprietary algorithm)
- Use client-side encryption when: proprietary/custom algorithm required by regulation

**Glue crawler minimum schedule = 5 minutes:**
- Cannot schedule a Glue crawler more frequently than every **5 minutes** (not 1 minute)
- For real-time schema updates: `S3:ObjectCreated:*` event → Lambda → invoke Glue crawler on-demand
- CloudWatch Events cannot directly target Glue crawlers as a destination type

**Lake Formation row-level security for country-based data access:**
- Lake Formation data filters: restrict rows by country + restrict columns → per-analyst fine-grained access
- S3 Access Points: cannot dynamically filter data rows based on requester identity
- Creating separate S3 buckets per country = high operational overhead + not scalable
- Migrating to per-region buckets = similar overhead + still doesn't enforce row-level access

**Redshift Spectrum + Lake Formation column-level access:**
- Create IAM role for Redshift with **Lake Formation access policy** (not direct S3 bucket policy)
- Create **external schema** in Redshift (not internal) to query S3 data via Spectrum
- Grant column-level permissions in Lake Formation (console or SQL GRANT)
- S3 bucket policy cannot enforce column-level restrictions (bucket policies are object-level only)

**S3 CRR pattern with Glacier Deep Archive for DR:**
- Production bucket: S3 Standard (immediate restore available)
- DR bucket: CRR replicates to S3 Standard → lifecycle rule immediately transitions to Glacier Deep Archive
- **Cannot use Standard-IA for immediate lifecycle transition** (30-day minimum storage charge applies)
- Glacier Deep Archive retrieval = 12 hours (acceptable for DR with 24-hour RTO requirement)
- Storing directly in Glacier Deep Archive in production = fails the "immediate restore" requirement

**QuickSight cross-region Redshift connectivity:**
- Create new **security group** on Redshift cluster (us-east-1) with inbound rule allowing QuickSight CIDR from ap-south-1
- VPC endpoints cannot span regions; NACLs are subnet-level (QuickSight may not be in same VPC)
- Each AWS region has documented QuickSight IP ranges for security group rules

## Domain 4 – Data Security and Governance (Set 7)

### RDS IAM Database Authentication — Support Matrix

```
IAM DB Auth Support:
  ✅ RDS MySQL
  ✅ RDS MariaDB
  ✅ RDS PostgreSQL
  ✅ Aurora MySQL
  ✅ Aurora PostgreSQL
  ❌ RDS Oracle
  ❌ RDS SQL Server
  ❌ RDS Db2  (Db2 is not even a supported RDS engine)
```
- Uses **auth token** (15-min lifetime) instead of password — no password stored in DB
- IAM manages authentication externally; DB users still need to exist in the database

---

### S3 Encryption Decision Guide

| Requirement | Use |
|---|---|
| Simplest, no key management | SSE-S3 (default) |
| Audit trail + no key management | **SSE-KMS** |
| Customer provides their own key | **SSE-C** |
| Proprietary encryption algorithm | Client-Side Encryption |

- **SSE-KMS**: CloudTrail logs every CMK use (who, when, what) — choose when regulation requires key usage audit
- **SSE-C**: AWS encrypts/decrypts using your key per request; AWS **never stores** your key
- **SSE-S3**: unique key per object by default — but no audit trail

---

### Redshift Audit Logging
- **NOT enabled by default** — must explicitly enable
- Logs: connection attempts, authentications, disconnections, queries, user activities
- Logs stored in **Amazon S3** bucket
- **CloudTrail**: logs only API calls (CreateCluster, DeleteCluster) — NOT DB-level activity
- **CloudWatch**: cluster performance metrics only — NOT audit logs
- **AWS Config**: compliance recording — NOT DB-level activity

---

### WAF Rule Combination

```
WebACL Priority Order (evaluated top to bottom):
┌─────────────────────────────────────────────────────────────────┐
│  Priority 1: IP Set ALLOW  → dev team IPs       → ALLOW ✅     │
│  Priority 2: Geo Match BLOCK → blocked countries → BLOCK ❌     │
│  Default Action:                                → ALLOW         │
└─────────────────────────────────────────────────────────────────┘

WAF can be attached to: CloudFront | ALB | API Gateway
Security Groups: can ALLOW traffic but CANNOT block SQLi/XSS patterns
NACLs: can DENY by IP/CIDR but CANNOT block geographic or SQLi/XSS patterns
```
- Geo match + IP set rules can coexist in one WebACL with priority ordering
- Application Load Balancer itself has **no** geo match or IP set statement capability

---

### S3 Object Lambda — PII Redaction

```
Without S3 Object Lambda:
  Client ──────────────────────────────▶ S3 Bucket (raw PII data)

With S3 Object Lambda:
  Client ──▶ S3 Object Lambda    ──▶ Lambda Function  ──▶ S3 Bucket
     ◀──────── Access Point        (redact/transform)
  [Modified/Redacted Response returned to client]
```
- Intercepts S3 **GET requests**; Lambda runs transformation before returning data
- Use cases: PII redaction, format conversion, watermarking, row/column filtering
- S3 Access Points: control **who** can access data — no row/column-level transformation
- S3 Gateway/Interface Endpoints: private access routing — no transformation capability

---

### RDS Read Replica vs Multi-AZ

| Feature | Read Replica | Multi-AZ Standby |
|---|---|---|
| Accessible for reads? | ✅ Yes | ❌ No (failover only) |
| Replication type | Async | Sync |
| Use for | Analytics offload | High availability |
| Cross-region possible? | ✅ Yes (extra cost) | ✅ Yes |
| Intra-region transfer cost | Free | Free |

- To offload analytics from production DB: **Read Replica in same region** (free data transfer)
- Multi-AZ standby is NOT accessible for queries — common exam trap

---

### DynamoDB DAX vs ElastiCache

| Feature | DAX | ElastiCache |
|---|---|---|
| API compatibility | Same as DynamoDB | Different API |
| Application refactoring | **None needed** | Required |
| Caching target | DynamoDB only | Any backend |
| Hot partition fix | ✅ Transparent | ✅ But requires code changes |

- Use **DAX** when you need to fix hot partition issues with minimal code changes
- DAX caches frequently accessed ("hot") keys in memory at microsecond latency

---

### Security Group Inbound Rule — Allowed Sources
```
VALID sources for SG inbound rules:
  ✅ IP address (e.g., 203.0.113.5/32)
  ✅ CIDR range (e.g., 10.0.0.0/16)
  ✅ Another Security Group ID

INVALID:
  ❌ Internet Gateway ID  ← common exam trap
  ❌ DENY rules (SGs are allow-only; use NACLs for DENY)
```

---

### EBS Root Volume Behavior on EC2 Termination
- Root EBS volume: **deleted on termination by default** (can be changed)
- Additional (non-root) EBS data volumes: **persist by default** after termination
- Instance store root volumes: **always lost** on stop/termination (no change possible)
- Setting: "Delete on Termination" flag per volume at launch time

---

### Redshift Cross-Region KMS Snapshot
- KMS keys are **region-specific** — cannot use source-region key in destination region
- Steps:
  1. Create **snapshot copy grant** in the **destination region** using a **destination region KMS key**
  2. Configure **cross-region snapshots** in the **source region** (reference the destination grant)
- Common traps: configuring snapshots in destination region (wrong); using source-region key in destination (impossible)

---

### Glacier Vault Lock (WORM)
- Use **vault lock policy** on an S3 Glacier vault to enforce compliance controls
- Specify WORM ("write once read many") → lock policy from future edits
- Vault lock policy ≠ ACL (ACLs manage access, not compliance enforcement)
- Vault lock policy ≠ S3 lifecycle policy (lifecycle = object transitions/deletions)

---

### VPC Gateway Endpoints — S3 and DynamoDB
- **Only S3 and DynamoDB** support gateway endpoints (route table entry)
- All other services use **interface endpoints** (private IP from subnet, not route table)
- DynamoDB also supports interface endpoints via AWS PrivateLink (but connected via private IP, not route table)
- Use gateway endpoints: no internet gateway, no NAT, no additional cost

---

### AWS Lake Formation — Fine-Grained Access Control
- Create **data filters** for row-level, column-level, and cell-level security
- Grant SELECT permission with data filter → users see only their allowed rows/columns
- Works with **Athena** for query-time enforcement on S3 data registered in Lake Formation
- IAM identity-based policies cannot provide fine-grained row/column filtering
- S3 Access Points cannot provide row/column-level access control
- QuickSight RLS requires Enterprise edition